# Notebook 2 - LSTM Parking Demand Forecaster

**Goal:** learn from historical parking occupancy and predict the next 8 half-hour slots (4 hours).

What this notebook does (everything is done here in Colab - the Streamlit app never trains):
1. Load history (your real CSV, or **clearly labelled synthetic data** if you have none)
2. Clean it (missing timestamps, duplicates, odd intervals, invalid values)
3. Convert to 30-minute intervals and create features
4. Split **chronologically** into train / validation / test
5. Train the LSTM and evaluate with MAE and RMSE (and compare with simple baselines)
6. Save `lstm_model.keras`, `scaler.pkl`, `lstm_config.json`

> The model predicts the **occupancy ratio** (occupied / capacity). Vacant spaces = capacity x (1 - ratio).

## Step 1 - Imports and settings
Change the settings in the second block if you need to.

In [ ]:
import os, json, random, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import joblib
import tensorflow as tf
from sklearn.preprocessing import MinMaxScaler

warnings.filterwarnings("ignore")
SEED = 42
random.seed(SEED); np.random.seed(SEED); tf.random.set_seed(SEED)
print("TensorFlow", tf.__version__)

In [ ]:
# ---------------- SETTINGS ----------------
MODEL_VERSION = "v1"            # name this model (use v2, v3 ... when you retrain)
CAPACITY = 60                   # parking spaces (only used to show errors in 'spaces' and to make synthetic data)
WINDOW = 48                     # model looks at the last 48 slots (24 hours)
HORIZON = 8                     # and predicts the next 8 slots (4 hours)
EPOCHS = 60
BATCH_SIZE = 64

HISTORY_CSV = "parking_history.csv"      # upload your own history to /content (columns: timestamp,occupied,capacity)
NEW_OBSERVATIONS_CSV = None              # retraining: e.g. "parking_observations_export.csv" from the Admin dashboard
CURRENT_MODEL_DIR = None                 # retraining: folder holding the OLD lstm_model.keras + scaler.pkl, to compare fairly
MAX_INTERP_GAP = 4                       # fill gaps up to 4 slots (2 hours) by interpolation; longer gaps stay empty

## Step 2 - Load the history

If `parking_history.csv` is not found, we **generate synthetic data**. It is fake, made only so the
pipeline can be tested. It deliberately contains dirty rows (missing, duplicated, invalid, off-grid timestamps)
so the cleaning step has something to do. A model trained on it learns the *synthetic pattern*, not a real car park -
the saved config records this (`trained_on_synthetic_data`).

In [ ]:
def generate_synthetic(days=60, capacity=CAPACITY, seed=SEED):
    """SYNTHETIC data: weekday office-style demand, lighter weekends, AR(1) noise, plus dirty rows."""
    rng = np.random.default_rng(seed)
    end = pd.Timestamp.now(tz="Asia/Kolkata").tz_localize(None).floor("30min") - pd.Timedelta(minutes=30)
    idx = pd.date_range(end=end, periods=days * 48, freq="30min")
    hour = idx.hour + idx.minute / 60
    weekday = 0.10 + 0.75 * np.exp(-((hour - 12.5) / 3.5) ** 2)
    weekend = 0.08 + 0.45 * np.exp(-((hour - 16.0) / 3.5) ** 2)
    base = np.where(idx.dayofweek >= 5, weekend, weekday)
    day_factor = pd.Series(rng.normal(1, 0.08, len(set(idx.date))), index=sorted(set(idx.date)))
    base = base * day_factor.reindex(idx.date).values
    noise = np.zeros(len(idx))
    for i in range(1, len(idx)):
        noise[i] = 0.8 * noise[i - 1] + rng.normal(0, 0.03)
    ratio = np.clip(base + noise, 0.02, 1.0)
    df = pd.DataFrame({"timestamp": idx, "occupied": np.round(ratio * capacity).astype(int),
                       "capacity": capacity, "source": "synthetic"})
    # --- make it dirty on purpose ---
    df = df.drop(df.sample(frac=0.02, random_state=1).index)                         # missing slots
    df = pd.concat([df, df.sample(frac=0.01, random_state=2)])                       # duplicates
    jitter = df.sample(frac=0.03, random_state=3).index
    df.loc[jitter[: len(jitter) // 2], "timestamp"] += pd.Timedelta(minutes=2)       # off-grid timestamps
    bad = pd.DataFrame({"timestamp": df["timestamp"].sample(6, random_state=4).values,
                        "occupied": [-5, capacity + 20, np.nan, 999, -1, np.nan],
                        "capacity": capacity, "source": "synthetic"})
    df = pd.concat([df, bad]).sample(frac=1, random_state=5).reset_index(drop=True)   # shuffled
    return df

if os.path.exists(HISTORY_CSV):
    raw = pd.read_csv(HISTORY_CSV)
    DATA_IS_SYNTHETIC = bool((raw.get("source", "") == "synthetic").all()) if "source" in raw else False
    print(f"Loaded {len(raw)} rows from {HISTORY_CSV}")
else:
    raw = generate_synthetic()
    DATA_IS_SYNTHETIC = True
    print(f"*** {HISTORY_CSV} not found -> using SYNTHETIC data ({len(raw)} rows). Results are NOT real parking behaviour. ***")

if "source" not in raw.columns:
    raw["source"] = "history_csv"
if NEW_OBSERVATIONS_CSV:
    new = pd.read_csv(NEW_OBSERVATIONS_CSV)
    raw = pd.concat([raw, new[["timestamp", "occupied", "capacity", "source"]]], ignore_index=True)
    DATA_IS_SYNTHETIC = DATA_IS_SYNTHETIC and bool((raw["source"] == "synthetic").all())
    print(f"Added {len(new)} rows from {NEW_OBSERVATIONS_CSV}")
raw.head()

## Step 3 - Look at the problems in the raw data

In [ ]:
tmp = raw.copy()
tmp["ts"] = pd.to_datetime(tmp["timestamp"], errors="coerce")
tmp["occupied"] = pd.to_numeric(tmp["occupied"], errors="coerce")
print("Rows:                       ", len(tmp))
print("Unreadable timestamps:      ", tmp["ts"].isna().sum())
print("Missing occupied/capacity:  ", tmp[["occupied", "capacity"]].isna().any(axis=1).sum())
print("Invalid (neg. or > capacity):", ((tmp["occupied"] < 0) | (tmp["occupied"] > tmp["capacity"])).sum())
print("Duplicate timestamps:       ", tmp["ts"].duplicated().sum())
print("Off the 30-min grid:        ", (tmp["ts"].dt.minute % 30 != 0).sum())

## Step 4 - Clean the data and convert to 30-minute intervals

Rules (each one is printed so you can explain it):
* drop rows with unreadable timestamps or missing counts
* drop impossible values (negative, or more cars than spaces, or capacity <= 0)
* snap timestamps to the 30-minute grid
* if two rows fall in the same slot keep the most trustworthy: `admin_verified` > `cnn_estimate` > others
* build a complete 30-minute timeline; interpolate only **short** gaps, leave long gaps empty (we do not invent days of data)

In [ ]:
df = raw.copy()
df["timestamp"] = pd.to_datetime(df["timestamp"], errors="coerce")
df["occupied"] = pd.to_numeric(df["occupied"], errors="coerce")
df["capacity"] = pd.to_numeric(df["capacity"], errors="coerce")
n_start = len(df)

df = df.dropna(subset=["timestamp", "occupied", "capacity"]);                       n1 = len(df)
df = df[(df["capacity"] > 0) & (df["occupied"] >= 0) & (df["occupied"] <= df["capacity"])]; n2 = len(df)
df["timestamp"] = df["timestamp"].dt.floor("30min")
priority = {"admin_verified": 0, "cnn_estimate": 1}
df["prio"] = df["source"].map(priority).fillna(2)
df = df.sort_values(["timestamp", "prio"]).drop_duplicates("timestamp", keep="first"); n3 = len(df)
df = df.drop(columns="prio").sort_values("timestamp").reset_index(drop=True)

print(f"start {n_start} -> after dropping missing {n1} -> after dropping invalid {n2} -> after de-duplicating {n3}")

clean_real = df.copy()                                   # real rows only - saved for the app
ratio = (df.set_index("timestamp")["occupied"] / df.set_index("timestamp")["capacity"])
full_index = pd.date_range(ratio.index.min(), ratio.index.max(), freq="30min")
ratio = ratio.reindex(full_index)
n_missing = int(ratio.isna().sum())
ratio = ratio.interpolate(limit=MAX_INTERP_GAP, limit_area="inside")
print(f"Timeline: {len(full_index)} slots, {n_missing} missing, {int(ratio.isna().sum())} still empty after interpolating gaps <= {MAX_INTERP_GAP} slots")
ratio.name = "occ_ratio"
ratio.plot(figsize=(12, 3), title="Occupancy ratio after cleaning"); plt.show()

## Step 5 - Features

Same function (same feature **order**) as `lstm_model.py` in the app:
`occ_ratio`, `occ_lag1` (previous slot), `occ_lag48` (same time yesterday), `hour_sin/cos`, `dow_sin/cos` (day of week), `is_weekend`.
Sine/cosine let the model know that 23:30 is close to 00:00.

In [ ]:
FEATURES = ["occ_ratio", "occ_lag1", "occ_lag48",
            "hour_sin", "hour_cos", "dow_sin", "dow_cos", "is_weekend"]

def make_features(ratio):
    """ratio: pandas Series (values 0-1) on a regular 30-minute DatetimeIndex.  (identical copy in lstm_model.py)"""
    df = pd.DataFrame({"occ_ratio": ratio.astype("float64")})
    df["occ_lag1"] = df["occ_ratio"].shift(1)       # previous slot
    df["occ_lag48"] = df["occ_ratio"].shift(48)     # same time yesterday
    hour = df.index.hour + df.index.minute / 60.0
    df["hour_sin"] = np.sin(2 * np.pi * hour / 24)
    df["hour_cos"] = np.cos(2 * np.pi * hour / 24)
    df["dow_sin"] = np.sin(2 * np.pi * df.index.dayofweek / 7)
    df["dow_cos"] = np.cos(2 * np.pi * df.index.dayofweek / 7)
    df["is_weekend"] = (df.index.dayofweek >= 5).astype("float64")
    return df[FEATURES]

feats = make_features(ratio)
feats.tail()

## Step 6 - Chronological split (no shuffling!)

Time series must be split by time: the **oldest 70%** trains, the **next 15%** validates (early stopping),
the **newest 15%** is the test set the model never sees. The scaler is fitted on the training part only.
Windows are built *inside* each part so no window straddles two parts.

In [ ]:
n = len(feats)
i_train, i_val = int(n * 0.70), int(n * 0.85)
parts = {"train": slice(0, i_train), "val": slice(i_train, i_val), "test": slice(i_val, n)}
for k, s in parts.items():
    print(f"{k:5s}: {feats.index[s][0]}  ->  {feats.index[s][-1]}   ({len(feats.iloc[s])} slots)")

scaler = MinMaxScaler()
scaler.fit(feats.iloc[parts["train"]].values)             # fit on TRAIN only

def make_windows(split):
    X_all = scaler.transform(feats.iloc[split].values)
    r = ratio.iloc[split].values
    X, y, t0 = [], [], []
    for i in range(len(X_all) - WINDOW - HORIZON + 1):
        xs, ys = X_all[i:i + WINDOW], r[i + WINDOW:i + WINDOW + HORIZON]
        if np.isnan(xs).any() or np.isnan(ys).any():
            continue                                       # skip windows touching long gaps
        X.append(xs); y.append(ys); t0.append(i + WINDOW)
    return np.array(X, "float32"), np.array(y, "float32"), np.array(t0), r

X_train, y_train, _, _ = make_windows(parts["train"])
X_val, y_val, _, _ = make_windows(parts["val"])
X_test, y_test, t0_test, r_test = make_windows(parts["test"])
print("windows  train", X_train.shape, " val", X_val.shape, " test", X_test.shape)
assert len(X_train) > 0 and len(X_val) > 0 and len(X_test) > 0, "Not enough clean data - need several weeks of history."

## Step 7 - Build and train the LSTM

In [ ]:
model = tf.keras.Sequential([
    tf.keras.layers.Input((WINDOW, len(FEATURES))),
    tf.keras.layers.LSTM(64, return_sequences=True),
    tf.keras.layers.Dropout(0.2),
    tf.keras.layers.LSTM(32),
    tf.keras.layers.Dense(32, activation="relu"),
    tf.keras.layers.Dense(HORIZON, activation="sigmoid"),     # HORIZON ratios between 0 and 1
])
model.compile(optimizer=tf.keras.optimizers.Adam(1e-3), loss="mae")
model.summary()

callbacks = [tf.keras.callbacks.EarlyStopping(patience=8, restore_best_weights=True, monitor="val_loss"),
             tf.keras.callbacks.ReduceLROnPlateau(patience=3, factor=0.5, monitor="val_loss")]
history = model.fit(X_train, y_train, validation_data=(X_val, y_val),
                    epochs=EPOCHS, batch_size=BATCH_SIZE, callbacks=callbacks, verbose=2)

In [ ]:
plt.figure(figsize=(7, 3.5))
plt.plot(history.history["loss"], label="training loss")
plt.plot(history.history["val_loss"], label="validation loss")
plt.xlabel("epoch"); plt.ylabel("MAE (ratio)"); plt.legend(); plt.title("LSTM training loss"); plt.show()

## Step 8 - Evaluate on the unseen test set

We report MAE and RMSE as ratio and in **parking spaces** (ratio x capacity).
Two simple baselines are included so the numbers mean something:
*persistence* (nothing changes) and *same time yesterday*. If the LSTM does not beat them, say so honestly.

In [ ]:
def mae(a, b): return float(np.mean(np.abs(a - b)))
def rmse(a, b): return float(np.sqrt(np.mean((a - b) ** 2)))

pred = model.predict(X_test, verbose=0)

# baselines on the same test windows
persist = np.repeat(r_test[t0_test - 1][:, None], HORIZON, axis=1)
yday = np.stack([r_test[t0_test + h - 48] for h in range(HORIZON)], axis=1)
valid = ~np.isnan(yday).any(axis=1) & (t0_test + 0 - 48 >= 0)

rows = [("LSTM", mae(pred, y_test), rmse(pred, y_test)),
        ("Persistence (baseline)", mae(persist, y_test), rmse(persist, y_test))]
if valid.any():
    rows.append(("Same time yesterday (baseline)", mae(yday[valid], y_test[valid]), rmse(yday[valid], y_test[valid])))
res = pd.DataFrame(rows, columns=["model", "MAE (ratio)", "RMSE (ratio)"])
res["MAE (spaces)"] = res["MAE (ratio)"] * CAPACITY
res["RMSE (spaces)"] = res["RMSE (ratio)"] * CAPACITY
display(res.round(3))

print("LSTM MAE by forecast step (spaces):")
print([round(mae(pred[:, h], y_test[:, h]) * CAPACITY, 2) for h in range(HORIZON)])
TEST_MAE_RATIO, TEST_RMSE_RATIO = rows[0][1], rows[0][2]

In [ ]:
# Actual vs predicted: 1 step ahead (30 min) and 4 steps ahead (2 hours) for the first 4 test days
t_index = feats.iloc[parts["test"]].index[t0_test]
k = min(len(t_index), 192)
fig, ax = plt.subplots(2, 1, figsize=(12, 6), sharex=True)
for a, h, title in zip(ax, [0, 3], ["30 minutes ahead", "2 hours ahead"]):
    a.plot(t_index[:k], y_test[:k, h] * CAPACITY, label="actual")
    a.plot(t_index[:k], pred[:k, h] * CAPACITY, label="LSTM predicted")
    a.set_ylabel("occupied spaces"); a.set_title(title); a.legend()
plt.tight_layout(); plt.show()

## Step 9 - (Retraining only) compare with the current model

Set `CURRENT_MODEL_DIR` in Step 1 to the folder with the **old** model. Both models are scored on the **same** unseen test
windows. Only replace the model in the app if the new one is better.

In [ ]:
if CURRENT_MODEL_DIR:
    old_model = tf.keras.models.load_model(os.path.join(CURRENT_MODEL_DIR, "lstm_model.keras"), compile=False)
    old_scaler = joblib.load(os.path.join(CURRENT_MODEL_DIR, "scaler.pkl"))
    Xo_all = old_scaler.transform(feats.iloc[parts["test"]].values)
    Xo = np.array([Xo_all[t - WINDOW:t] for t in t0_test], "float32")
    old_pred = old_model.predict(Xo, verbose=0)
    old_mae = mae(old_pred, y_test) * CAPACITY
    new_mae = TEST_MAE_RATIO * CAPACITY
    print(f"OLD model test MAE: {old_mae:.2f} spaces | NEW model test MAE: {new_mae:.2f} spaces")
    print("=> NEW model is better - OK to replace." if new_mae < old_mae else "=> NEW model is NOT better - keep the old one.")
else:
    print("CURRENT_MODEL_DIR not set - skipping (first training).")

## Step 10 - Save the files the Streamlit app needs

* `lstm_model.keras` - the network
* `scaler.pkl` - the fitted scaler (same scaling at inference)
* `lstm_config.json` - window, horizon, feature order, version and honest metadata
* `parking_history.csv` - the **cleaned real rows** (copy into `data/`; the app uses it as history)

In [ ]:
os.makedirs("artifacts", exist_ok=True)
model.save("artifacts/lstm_model.keras")
joblib.dump(scaler, "artifacts/scaler.pkl")
config = {
    "version": MODEL_VERSION,
    "window": WINDOW, "horizon": HORIZON, "slot_minutes": 30,
    "features": FEATURES,
    "target": "occupancy_ratio (occupied / capacity) for the next HORIZON slots",
    "capacity_used_for_metrics": CAPACITY,
    "test_mae_ratio": TEST_MAE_RATIO, "test_rmse_ratio": TEST_RMSE_RATIO,
    "test_mae_spaces": TEST_MAE_RATIO * CAPACITY, "test_rmse_spaces": TEST_RMSE_RATIO * CAPACITY,
    "trained_on_synthetic_data": bool(DATA_IS_SYNTHETIC),
    "train_period": [str(feats.index[parts["train"]][0]), str(feats.index[parts["train"]][-1])],
    "test_period": [str(feats.index[parts["test"]][0]), str(feats.index[parts["test"]][-1])],
    "trained_at": str(pd.Timestamp.now()),
}
with open("artifacts/lstm_config.json", "w") as f:
    json.dump(config, f, indent=2)
clean_real[["timestamp", "occupied", "capacity", "source"]].to_csv("artifacts/parking_history.csv", index=False)

# sanity check: reload the saved files and make sure predictions are identical
m2 = tf.keras.models.load_model("artifacts/lstm_model.keras", compile=False)
s2 = joblib.load("artifacts/scaler.pkl")
assert np.allclose(m2.predict(X_test[:5], verbose=0), pred[:5], atol=1e-5), "reloaded model differs!"
assert json.load(open("artifacts/lstm_config.json"))["features"] == FEATURES
print("Saved and verified:", os.listdir("artifacts"))
print(json.dumps({k: config[k] for k in ["version", "test_mae_spaces", "trained_on_synthetic_data"]}, indent=2))

## Step 11 - Download
Runs in Colab and downloads the four files. Put the three model files in `models/` and the CSV in `data/` of your project.

In [ ]:
try:
    from google.colab import files
    for name in ["lstm_model.keras", "scaler.pkl", "lstm_config.json", "parking_history.csv"]:
        files.download(os.path.join("artifacts", name))
except ImportError:
    print("Not running in Colab - files are in the 'artifacts' folder.")

### Retraining checklist
1. Admin dashboard -> section C -> download the observations CSV
2. Upload it and your old `parking_history.csv` to Colab; set `NEW_OBSERVATIONS_CSV`, `MODEL_VERSION = "v2"`, and `CURRENT_MODEL_DIR` (upload the old model files into a folder)
3. Run all cells; check Step 9 says the new model is better
4. Replace the files in `models/`, then register the new version in the Admin dashboard

*CNN observations are estimates, not ground truth.* Training on them teaches the LSTM the CNN's mistakes too.
Prefer rows with `source = admin_verified` when you have enough of them.